# Financial Dashboard Analysis: Tesla vs GameStop

## Project Overview

This notebook extends the original IBM-style stock and revenue exercise into a cleaner finance dashboard project. It compares Tesla (`TSLA`) and GameStop (`GME`) using historical stock prices, company revenue observations, return metrics, volatility, drawdowns, and interactive Plotly visualizations.

The goal is exploratory financial analysis, not investment advice. The analysis focuses on what stock prices and revenue data can show together, while recognizing that market prices also reflect expectations, liquidity, sentiment, and risk.


## Data Sources and Methodology

- Stock prices are collected with `yfinance` from Yahoo Finance.
- Tesla revenue data is scraped from the IBM course-hosted revenue page.
- GameStop revenue data is scraped from the IBM course-hosted stock page.
- Revenue values are cleaned by removing currency symbols and commas, then converted to numeric values in millions of U.S. dollars.
- Market metrics are calculated from daily closing prices.
- Sharpe ratio uses a 0% risk-free rate for simplicity.

The sample period begins when both companies have overlapping price data, so Tesla and GameStop can be compared on the same timeline.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from io import StringIO
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.float_format = "{:,.4f}".format


## Stock Price Data Collection

The stock data is downloaded for Tesla and GameStop, then combined into one tidy DataFrame with a `Ticker` column.


In [2]:
TICKERS = {
    "Tesla": "TSLA",
    "GameStop": "GME",
}


def get_stock_data(company, ticker):
    data = yf.Ticker(ticker).history(period="max", auto_adjust=False).reset_index()
    data = data[["Date", "Open", "High", "Low", "Close", "Volume"]].copy()
    data["Date"] = pd.to_datetime(data["Date"], utc=True).dt.tz_convert(None)
    data["Company"] = company
    data["Ticker"] = ticker
    return data


stock_data = pd.concat(
    [get_stock_data(company, ticker) for company, ticker in TICKERS.items()],
    ignore_index=True,
)

common_start = stock_data.groupby("Ticker")["Date"].min().max()
stock_data = stock_data[stock_data["Date"] >= common_start].copy()
stock_data.head()


,Date,Open,High,Low,Close,Volume,Company,Ticker
0,2010-06-29 04:00:00,1.2667,1.6667,1.1693,1.5927,281494500,Tesla,TSLA
1,2010-06-30 04:00:00,1.7193,2.0280,1.5533,1.5887,257806500,Tesla,TSLA
2,2010-07-01 04:00:00,1.6667,1.7280,1.3513,1.4640,123282000,Tesla,TSLA
3,2010-07-02 04:00:00,1.5333,1.5400,1.2473,1.2800,77097000,Tesla,TSLA
4,2010-07-06 04:00:00,1.3333,1.3333,1.0553,1.0740,103003500,Tesla,TSLA


## Revenue Data Collection

Revenue data is scraped from the IBM course-hosted HTML pages using `requests`, `BeautifulSoup`, and `pandas.read_html`.


In [3]:
TESLA_REVENUE_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
GME_REVENUE_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"


def extract_quarterly_revenue(url, company, table_label):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tables = pd.read_html(StringIO(str(soup)))

    selected_table = None
    for table in tables:
        table_text = " ".join(map(str, table.columns)) + " " + table.head().to_string()
        if table_label in table_text:
            selected_table = table.copy()
            break

    if selected_table is None:
        raise ValueError(f"Could not find revenue table containing: {table_label}")

    revenue = selected_table.iloc[:, :2].copy()
    revenue.columns = ["Date", "Revenue"]
    revenue["Company"] = company
    return revenue


tesla_revenue_raw = extract_quarterly_revenue(
    TESLA_REVENUE_URL,
    "Tesla",
    "Tesla Quarterly Revenue",
)

gme_revenue_raw = extract_quarterly_revenue(
    GME_REVENUE_URL,
    "GameStop",
    "GameStop Quarterly Revenue",
)

revenue_raw = pd.concat([tesla_revenue_raw, gme_revenue_raw], ignore_index=True)
revenue_raw.head()


,Date,Revenue,Company
0,2022-09-30,"$21,454",Tesla
1,2022-06-30,"$16,934",Tesla
2,2022-03-31,"$18,756",Tesla
3,2021-12-31,"$17,719",Tesla
4,2021-09-30,"$13,757",Tesla


## Data Cleaning

The price and revenue datasets are converted to consistent date and numeric formats. Revenue is stated in millions of U.S. dollars.


In [4]:
revenue_data = revenue_raw.copy()
revenue_data["Date"] = pd.to_datetime(revenue_data["Date"], errors="coerce")
revenue_data["Revenue"] = (
    revenue_data["Revenue"]
    .astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .replace("", np.nan)
)
revenue_data["Revenue"] = pd.to_numeric(revenue_data["Revenue"], errors="coerce")
revenue_data = revenue_data.dropna(subset=["Date", "Revenue"]).sort_values(["Company", "Date"])

price_data = stock_data.sort_values(["Ticker", "Date"]).copy()
price_data["Daily Return"] = price_data.groupby("Ticker")["Close"].pct_change()
price_data["Cumulative Return"] = price_data.groupby("Ticker")["Daily Return"].transform(
    lambda returns: (1 + returns.fillna(0)).cumprod() - 1
)
price_data["Normalized Price"] = price_data.groupby("Ticker")["Close"].transform(lambda close: close / close.iloc[0] * 100)
price_data["Rolling 30D Volatility"] = price_data.groupby("Ticker")["Daily Return"].transform(
    lambda returns: returns.rolling(30).std() * np.sqrt(252)
)
price_data["Running Max"] = price_data.groupby("Ticker")["Close"].cummax()
price_data["Drawdown"] = price_data["Close"] / price_data["Running Max"] - 1

price_data.head()


,Date,Open,High,Low,Close,Volume,Company,Ticker,Daily Return,Cumulative Return,Normalized Price,Rolling 30D Volatility,Running Max,Drawdown
6100,2010-06-29 04:00:00,4.6400,4.6400,4.4900,4.5800,27437600,GameStop,GME,NaN,0.0000,100.0000,NaN,4.5800,0.0000
6101,2010-06-30 04:00:00,4.5875,4.7200,4.5600,4.6975,32174400,GameStop,GME,0.0257,0.0257,102.5655,NaN,4.6975,0.0000
6102,2010-07-01 04:00:00,4.6700,4.8325,4.6625,4.7675,39145600,GameStop,GME,0.0149,0.0409,104.0939,NaN,4.7675,0.0000
6103,2010-07-02 04:00:00,4.7550,4.7950,4.5600,4.5675,22867200,GameStop,GME,-0.0420,-0.0027,99.7271,NaN,4.7675,-0.0420
6104,2010-07-06 04:00:00,4.6000,4.7100,4.5625,4.6000,16960800,GameStop,GME,0.0071,0.0044,100.4367,NaN,4.7675,-0.0351


## Market Return Analysis

Daily returns show short-term price movement, while cumulative returns show total growth of one invested dollar over the shared sample period.


In [5]:
def calculate_metrics(data, revenue):
    rows = []
    for company, ticker in TICKERS.items():
        company_prices = data[data["Ticker"] == ticker].dropna(subset=["Close"]).copy()
        company_returns = company_prices["Daily Return"].dropna()
        company_revenue = revenue[revenue["Company"] == company].sort_values("Date")

        start_price = company_prices["Close"].iloc[0]
        latest_price = company_prices["Close"].iloc[-1]
        total_return = latest_price / start_price - 1
        years = (company_prices["Date"].iloc[-1] - company_prices["Date"].iloc[0]).days / 365.25
        annualized_return = (1 + total_return) ** (1 / years) - 1
        annualized_volatility = company_returns.std() * np.sqrt(252)
        max_drawdown = company_prices["Drawdown"].min()
        sharpe_ratio = annualized_return / annualized_volatility if annualized_volatility else np.nan
        latest_revenue = company_revenue["Revenue"].iloc[-1] if not company_revenue.empty else np.nan
        latest_revenue_date = company_revenue["Date"].iloc[-1] if not company_revenue.empty else pd.NaT

        rows.append({
            "Company": company,
            "Ticker": ticker,
            "Latest Price": latest_price,
            "Total Return": total_return,
            "Annualized Return": annualized_return,
            "Annualized Volatility": annualized_volatility,
            "Maximum Drawdown": max_drawdown,
            "Sharpe Ratio (0% RF)": sharpe_ratio,
            "Latest Revenue Observation": latest_revenue,
            "Latest Revenue Date": latest_revenue_date,
        })

    return pd.DataFrame(rows)


metrics = calculate_metrics(price_data, revenue_data)
metrics


,Company,Ticker,Latest Price,Total Return,Annualized Return,Annualized Volatility,Maximum Drawdown,Sharpe Ratio (0% RF),Latest Revenue Observation,Latest Revenue Date
0,Tesla,TSLA,430.1200,269.0627,0.4231,0.5742,-0.7363,0.7367,"21,454.0000",2022-09-30
1,GameStop,GME,22.6050,3.9356,0.1058,0.9619,-0.9514,0.1100,"1,021.0000",2020-04-30


In [6]:
def format_kpi_table(metrics):
    formatted = metrics.copy()
    formatted["Latest Price"] = formatted["Latest Price"].map("${:,.2f}".format)
    for column in ["Total Return", "Annualized Return", "Annualized Volatility", "Maximum Drawdown"]:
        formatted[column] = formatted[column].map("{:.2%}".format)
    formatted["Sharpe Ratio (0% RF)"] = formatted["Sharpe Ratio (0% RF)"].map("{:.2f}".format)
    formatted["Latest Revenue Observation"] = formatted["Latest Revenue Observation"].map("${:,.0f}M".format)
    formatted["Latest Revenue Date"] = formatted["Latest Revenue Date"].dt.strftime("%Y-%m-%d")
    return formatted


kpi_table = format_kpi_table(metrics)
kpi_table


,Company,Ticker,Latest Price,Total Return,Annualized Return,Annualized Volatility,Maximum Drawdown,Sharpe Ratio (0% RF),Latest Revenue Observation,Latest Revenue Date
0,Tesla,TSLA,$430.12,26906.27%,42.31%,57.42%,-73.63%,0.74,"$21,454M",2022-09-30
1,GameStop,GME,$22.60,393.56%,10.58%,96.19%,-95.14%,0.11,"$1,021M",2020-04-30


## Risk and Volatility Analysis

Annualized volatility is calculated from daily returns. A rolling 30-day volatility chart helps show when market risk increased, especially during sharp repricing periods.


In [7]:
fig = go.Figure()
for company, ticker in TICKERS.items():
    company_prices = price_data[price_data["Ticker"] == ticker]
    fig.add_trace(go.Scatter(
        x=company_prices["Date"],
        y=company_prices["Normalized Price"],
        mode="lines",
        name=company,
    ))

fig.update_layout(
    title="Tesla vs GameStop Normalized Price (Start = 100)",
    xaxis_title="Date",
    yaxis_title="Normalized Price",
    hovermode="x unified",
    height=500,
)
fig.show()


In [8]:
fig = go.Figure()
for company, ticker in TICKERS.items():
    company_prices = price_data[price_data["Ticker"] == ticker]
    fig.add_trace(go.Scatter(
        x=company_prices["Date"],
        y=company_prices["Cumulative Return"],
        mode="lines",
        name=company,
    ))

fig.update_layout(
    title="Tesla vs GameStop Cumulative Return",
    xaxis_title="Date",
    yaxis_title="Cumulative Return",
    yaxis_tickformat=".0%",
    hovermode="x unified",
    height=500,
)
fig.show()


In [9]:
fig = go.Figure()
for company, ticker in TICKERS.items():
    company_prices = price_data[price_data["Ticker"] == ticker]
    fig.add_trace(go.Scatter(
        x=company_prices["Date"],
        y=company_prices["Rolling 30D Volatility"],
        mode="lines",
        name=company,
    ))

fig.update_layout(
    title="Rolling 30-Day Annualized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    yaxis_tickformat=".0%",
    hovermode="x unified",
    height=500,
)
fig.show()


## Drawdown Analysis

Drawdown measures the percentage decline from a previous peak. This helps compare downside risk during major selloffs.


In [10]:
fig = go.Figure()
for company, ticker in TICKERS.items():
    company_prices = price_data[price_data["Ticker"] == ticker]
    fig.add_trace(go.Scatter(
        x=company_prices["Date"],
        y=company_prices["Drawdown"],
        mode="lines",
        name=company,
    ))

fig.update_layout(
    title="Drawdown from Previous Closing Price High",
    xaxis_title="Date",
    yaxis_title="Drawdown",
    yaxis_tickformat=".0%",
    hovermode="x unified",
    height=500,
)
fig.show()


## Revenue Trend Analysis

Revenue data provides operating context, but it does not fully explain stock prices. Prices can move ahead of fundamentals because they reflect expectations, liquidity, sentiment, risk appetite, and investor positioning.


In [11]:
fig = go.Figure()
for company in TICKERS.keys():
    company_revenue = revenue_data[revenue_data["Company"] == company]
    fig.add_trace(go.Bar(
        x=company_revenue["Date"],
        y=company_revenue["Revenue"],
        name=company,
    ))

fig.update_layout(
    title="Quarterly Revenue Comparison",
    xaxis_title="Date",
    yaxis_title="Revenue ($US Millions)",
    barmode="group",
    hovermode="x unified",
    height=500,
)
fig.show()


## Interactive Financial Dashboard

The dashboard combines price, cumulative return, volatility, drawdown, and revenue into one view for comparison.


In [12]:
fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Closing Price",
        "Cumulative Return",
        "Rolling 30-Day Volatility",
        "Drawdown",
        "Quarterly Revenue",
        "KPI Summary",
    ),
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "table"}],
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
)

for company, ticker in TICKERS.items():
    company_prices = price_data[price_data["Ticker"] == ticker]
    company_revenue = revenue_data[revenue_data["Company"] == company]

    fig.add_trace(go.Scatter(x=company_prices["Date"], y=company_prices["Close"], mode="lines", name=f"{company} Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=company_prices["Date"], y=company_prices["Cumulative Return"], mode="lines", name=f"{company} Return"), row=1, col=2)
    fig.add_trace(go.Scatter(x=company_prices["Date"], y=company_prices["Rolling 30D Volatility"], mode="lines", name=f"{company} Volatility"), row=2, col=1)
    fig.add_trace(go.Scatter(x=company_prices["Date"], y=company_prices["Drawdown"], mode="lines", name=f"{company} Drawdown"), row=2, col=2)
    fig.add_trace(go.Bar(x=company_revenue["Date"], y=company_revenue["Revenue"], name=f"{company} Revenue"), row=3, col=1)

fig.add_trace(
    go.Table(
        header=dict(values=list(kpi_table.columns), fill_color="#1f2937", font=dict(color="white", size=11), align="left"),
        cells=dict(values=[kpi_table[column] for column in kpi_table.columns], fill_color="#f3f4f6", align="left", font=dict(size=10)),
    ),
    row=3,
    col=2,
)

fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
fig.update_yaxes(title_text="Return", tickformat=".0%", row=1, col=2)
fig.update_yaxes(title_text="Volatility", tickformat=".0%", row=2, col=1)
fig.update_yaxes(title_text="Drawdown", tickformat=".0%", row=2, col=2)
fig.update_yaxes(title_text="Revenue ($US Millions)", row=3, col=1)

fig.update_layout(
    title="Tesla vs GameStop Financial Dashboard",
    height=1200,
    hovermode="x unified",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.show()


## Key Findings

- Tesla shows a growth-stock profile where revenue expansion and market repricing are both visible. Its stock performance reflects not only reported revenue growth, but also changing investor expectations about future scale, margins, and industry leadership.
- GameStop shows a speculative meme-stock episode where market price temporarily disconnected from underlying revenue trends. The price spike is more visible in return, volatility, and drawdown metrics than in the revenue series.
- Stock prices reflect expectations, liquidity, sentiment, and risk, not only current revenue. Revenue is useful context, but it is not a complete valuation model by itself.
- The KPI summary highlights the tradeoff between return and risk: strong upside periods can come with high volatility and large drawdowns.


## Limitations and Possible Extensions

- This notebook is educational and exploratory; it is not financial advice.
- Revenue data is scraped from static course-hosted HTML pages and may not reflect the most recent company filings.
- Stock data comes from `yfinance`, which depends on Yahoo Finance availability and formatting.
- The Sharpe ratio uses a 0% risk-free rate for simplicity.
- Metrics are based on historical closing prices and do not forecast future returns.
- Possible extensions include adding benchmark comparisons, sector peers, inflation or interest-rate context, and more formal valuation ratios if reliable data sources are added.
